### 1. 임베딩
- 텍스트를 문장 단위로 벡터 : [0.25, 0.12, 0.11 ... ] 이런 식으로 변환
- 검색, 추천 시스템, RAG(챗봇), 분류 -> 이런 벡터 변환 사용

### 2. 임베딩 모델 준비
- 한국어에 강한 최신 모델 - 작고 빠른 걸로 -> cpu도 가능한

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("dragonkue/multilingual-e5-small-ko-v2")
print("모델 준비 완료")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\study-with-ai-2\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--dragonkue--multilingual-e5-small-ko-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/23.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

모델 준비 완료


### E5 - 모델 사용법 익히기
- query : 질문 입력할 때
- passage : 검색 대상이 되는 문서

In [2]:
sentence = "오늘 점심 메뉴는?"

with_prefix = model.encode("query: "+ sentence, normalize_embeddings=True)  # 뒤에거 True 해야 유사도 계산 잘됨
without_prefix = model.encode(sentence, normalize_embeddings=True)  # 뒤에거 True 해야 유사도 계산 잘됨

print("접두사가 있을 때", with_prefix[:5])
print("접두사가 없을 때", without_prefix[:5])

접두사가 있을 때 [ 0.02129692 -0.0158352  -0.09245932  0.01726908  0.07143722]
접두사가 없을 때 [ 0.02251527 -0.00308811 -0.0859407   0.00292922  0.088636  ]


In [5]:
# sentences = ["오늘 점심으로 무엇을 먹을까요?",
#              "점심 메뉴를 추천해주세요",
#              "오늘 점심은 김치찌개가 어때요",
#              "저녁에는 어떤 거 먹을까요",
#              "오늘 영화 보러 갈까요?",
#              "파이썬 공부 너무 재밌어요",
#              "인공지능은 너무 재밌어요"]


(7, 384)

In [6]:
# 기사 내용을 통째로 벡터로 바꾸면
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv")

article_vec = model.encode("passage :" + df['정제본문'][0], normalize_embeddings=True)
article_vec[:5]

array([ 0.06202456,  0.02759574, -0.04063224,  0.00485639,  0.01837734],
      dtype=float32)

In [ ]:
df['passage_add'] = df['정제본문'].apply(lambda x : "passage :" + x)    # 먼저 본문 내용에 전부 "passage :" 추가하기

In [20]:
article_vecs = model.encode(df['passage_add'].tolist()[:100], normalize_embeddings=True)
article_vecs.shape

(100, 384)

In [35]:
from sentence_transformers import util

def search_text(query, k=3):    # 질문과 유사한 문서 최대 k 개 가져오기
    query_vec = model.encode("query: " + query, normalize_embeddings=True)
    similarity = util.cos_sim(query_vec, article_vecs)[0]
    for i in similarity.argsort(descending=True)[:k]:   #유사도 상위 k개만 뽑음
        i = int(i)
        print(f"{similarity[i]}, {df['정제본문'].iloc[i]}")

In [36]:
query = "경제 상황이 어떻게 되고 있어?"
search_text(query)

0.5671099424362183, 경제와이드 모닝벨 조간 브리핑 장연재 조간브리핑입니다 발길 끊는 개미들 거래대금 20개월만에 최저 동학개미들이 국내 주식시장을 떠나고 있다는 기사 먼저 보겠습니다 중앙일보입니다 미국발 금리 인상과 세계 경제 침체 우려로 올해에만 코스피가 20 넘게 급락하자 개인투자자들이 증시를 이탈하고 있습니다 코스피에서 개인 투자자의 하루 평균 거래 대금은 4조 3 009억 원으로 2년 4개월 만에 최저 수준을 기록했습니다 증시 대기자금인 투자자예탁금은 지난달 말 기준 반년 만에 10조 원이 줄었고 빚을 내 투자한 후 아직 갚지 않은 금액인 신용거래융자 잔고도 같은 기간 5조 원 이상 줄었습니다 이런 가운데 당분간 전 세계의 경기 침체 공포로 국내증시 반등을 기대하긴 어렵다는 의견이 지배적인데요 반면 일부 증권가에서는 기존 악재가 반영된 만큼 약세장을 오히려 기회로 삼아야 한다는 의견도 있습니다 지방 중심 역전세 깡통아파트 확산 경향신문 기사입니다 최근 지방의 저가 아파트를 중심으로 전세가격이 매매가격보다 높은 역전세 현상이 나타나고 있습니다 정부의 다주택자 규제 완화 조치 이후 매물이 쌓이고 있지만 금리가 오르자 집을 사려는 사람이 없어 집값은 하락하고 있습니다 반면 전세시장은 신규 계약체결 시점에 경신까지 염두에 두고 4년 치 상승분을 미리 가격에 반영하려는 경향이 있어서 역전세가 발생하고 있는 건데요 전세계약 시점에 집값이 이미 전세가격보다 낮다면 보증금을 보호받을 수 없어서 문제가 되고 있습니다 이런 현상은 현재까지 갭투자가 주로 이뤄지는 3억 원 이하 지방 저가 아파트에서 주로 나타나는데요 전문가들은 서울 집값은 큰 변화가 없기 때문에 전국적인 문제로 보기는 어렵다고 분석했습니다 경기 침체에 빅테크 감원 시작됐다 전 세계 기업에 감원 공포가 불고 있다는 조선일보 기사도 보겠습니다 코로나 팬데믹 기간 큰돈을 벌며 조직 규모를 키웠던 테크 기업들 너도나도 긴축 경영에 돌입하고 있습니다 인플레이션과 금리 인상 등으로 사업 환경이 급속도

In [37]:
query = "금리 인상 관련해서 알려줘"
search_text(query, 5)

0.5943858623504639, 예대금리차 7년 7개월만에 가장 큰 폭으로 늘어 은행들 여론 악화 부담 느껴 이미지출처 연합뉴스 정치권과 금융당국의 이자장사 경고에 시중은행들이 대출금리를 계속 낮추고 정기 예 적금 상품의 금리는 특판 등을 통해 연 3 5 대까지 올리고 있다 대출금리와 예대금리 간의 차이인 의미하는 예대금리차가 7년 7개월 만에 가장 큰 폭으로 벌어지자 여론이 악화돼 은행들이 부담을 느낀 것도 이런 조치를 내린 이유다 3일 금융권에 따르면 신한은행은 이르면 이번주 4 8일 부터 신규 취급 주택담보대출과 전세자금대출 금리를 각각 최대 0 35 포인트 p 0 30 p 내리기로 했다 취약 차주 프로그램 도 이달 초 시작할 예정이다 우선 6월말 기준 연 5 가 넘는 금리로 주택담보대출을 이용하는 차주의 금리를 1년간 연 5 로 일괄 인하하고 5 초과분은 은행이 대신 지원해주기로 했다 금리상한형 주택담보대출 연간 금리 상승폭 0 75 포인트 이내로 제한한 상품 을 신청하는 대출자에게는 원래 고객이 부담해야하는 연 0 2 p의 가산금리를 신한은행이 1년간 내주기로 했다 전세자금대출자 중 연소득 4000만원 이하 전세보증금 3억원 이하 의 조건을 갖춘 이들을 대상으로 금융채 2년물 금리를 기준으로 삼는 전세자금대출 상품도 내놓는다 전세자금대출은 일보통 6개월 또는 1년 단위 변동금리 상품인데 사실상 2년 단위 고정금리 상품을 내놔 금리 상승에 따른 위험을 낮추겠다는 의미다 신한은행은 더불어 대표적 서민 지원 금융상품인 새희망홀씨 대출의 신규 금리도 연 0 5 p 내릴 방침이다 NH농협은행은 이달 1일부터 우대금리 확대 등을 통해 담보 전세자금 등 주택관련대출 금리를 0 1 0 2 p 낮췄다 우리은행 역시 지난달 24일부터 은행채 5년물 기준 고정금리 대출에 적용하던 1 3 p의 우대금리 은행 자체 신용등급 7등급 이내 를 모든 등급 8 10등급 추가 에 한꺼번에 주기로 했다 우리은행 전체 등급의 가산금리가 1 5 p씩 낮아진 것과 마찬가지다 케이뱅크도 

### 실습
- 1. 내가 질문하려는 주제와 관련있는 데이터를 먼저 만들어보기
    - 예) 점심 메뉴 추천?
    - 리스트에 점심 메뉴 관련 데이터를 데이터프레임 형태로 만들어서 벡터화시켜야 함
- 2. 질문을 해서 얼마나 잘 찾는지 확인해보기


In [50]:
sentences = [
      "아메리카노는 에스프레소에 물을 더한 커피로, 뜨겁게 또는 차갑게 주문할 수 있으며 단맛이 거의 없습니다.",
      "카페라테는 에스프레소와 우유를 섞은 부드러운 커피이며, 따뜻한 라테와 아이스 라테를 모두 제공합니다.",
      "바닐라 라테는 에스프레소, 우유, 바닐라 시럽으로 만든 달콤한 커피 음료이며 따뜻하거나 차갑게 마실 수 있습니다.",
      "카라멜 마키아토는 우유와 바닐라 시럽 위에 에스프레소와 카라멜 소스를 올린 달콤한 따뜻한 커피입니다.",
      "콜드브루는 찬물로 오래 추출한 아이스 커피로, 산미가 적고 깔끔하며 시럽을 넣지 않아 달지 않습니다.",
      "카페모카는 에스프레소, 초콜릿 소스, 우유를 섞은 달콤한 커피 음료로 아이스와 핫 중 선택할 수 있습니다.",
      "디카페인 아메리카노는 카페인 부담을 줄인 커피로, 진하지만 달지 않은 맛을 원할 때 좋습니다.",
      "말차 라테는 말차 가루와 우유를 섞은 논커피 음료로, 쌉싸름하면서도 부드럽고 따뜻하게 또는 차갑게 주문할 수 있습니다.",
      "초콜릿 라테는 진한 초콜릿과 우유로 만든 카페인 없는 달콤한 음료이며 따뜻한 겨울 음료로 좋습니다.",
      "아이스 초코는 차가운 우유와 초콜릿 소스를 섞은 달콤한 논커피 음료로, 더운 날 시원하게 마시기 좋습니다.",
      "고구마 라테는 고구마의 달콤하고 고소한 맛을 살린 따뜻한 논커피 음료이며 카페인이 없습니다.",
      "밀크티는 홍차와 우유를 섞은 부드러운 음료로, 달콤한 아이스 밀크티 또는 따뜻한 밀크티로 주문할 수 있습니다.",
      "레몬 에이드는 레몬청과 탄산수를 섞은 차가운 논커피 음료로, 새콤달콤하고 상쾌한 맛이 특징입니다.",
      "자몽 허니 블랙티는 자몽과 꿀, 홍차를 넣은 달콤새콤한 논커피 음료로 아이스와 핫 모두 가능합니다.",
      "캐모마일 티는 카페인이 없는 허브차로, 달지 않고 편안한 향을 즐기고 싶을 때 따뜻하게 마시기 좋습니다."
  ]


In [46]:
cafe_vec = model.encode(["passage: " + sentence for sentence in sentences], normalize_embeddings=True)

def search_text(query, k=3):    # 질문과 유사한 문서 최대 k 개 가져오기
    query_vec = model.encode("query: " + query, normalize_embeddings=True)
    similarity = util.cos_sim(query_vec, cafe_vec)[0]
    for i in similarity.argsort(descending=True)[:k]:   #유사도 상위 k개만 뽑음
        i = int(i)
        print(f"{similarity[i]}, {sentences[i]}")

In [47]:
query1 = "카페인이 없고 따뜻한 음료 추천해 줘"
search_text(query1)

0.6973623633384705, 고구마 라테는 고구마의 달콤하고 고소한 맛을 살린 따뜻한 논커피 음료이며 카페인이 없습니다.
0.684100329875946, 초콜릿 라테는 진한 초콜릿과 우유로 만든 카페인 없는 달콤한 음료이며 따뜻한 겨울 음료로 좋습니다.
0.6817446947097778, 캐모마일 티는 카페인이 없는 허브차로, 달지 않고 편안한 향을 즐기고 싶을 때 따뜻하게 마시기 좋습니다.


In [51]:
query2 = "차갑고 달지 않은 커피가 뭐야?"
search_text(query2)

0.6433620452880859, 아메리카노는 에스프레소에 물을 더한 커피로, 뜨겁게 또는 차갑게 주문할 수 있으며 단맛이 거의 없습니다.
0.6342044472694397, 콜드브루는 찬물로 오래 추출한 아이스 커피로, 산미가 적고 깔끔하며 시럽을 넣지 않아 달지 않습니다.
0.632789671421051, 바닐라 라테는 에스프레소, 우유, 바닐라 시럽으로 만든 달콤한 커피 음료이며 따뜻하거나 차갑게 마실 수 있습니다.


In [52]:
query3 = "달콤한 초콜릿 음료를 마시고 싶어"
search_text(query3)

0.7244968414306641, 초콜릿 라테는 진한 초콜릿과 우유로 만든 카페인 없는 달콤한 음료이며 따뜻한 겨울 음료로 좋습니다.
0.6880515813827515, 아이스 초코는 차가운 우유와 초콜릿 소스를 섞은 달콤한 논커피 음료로, 더운 날 시원하게 마시기 좋습니다.
0.6682391166687012, 카페모카는 에스프레소, 초콜릿 소스, 우유를 섞은 달콤한 커피 음료로 아이스와 핫 중 선택할 수 있습니다.


In [55]:
query4 = "따뜻한 티"
search_text(query4)

0.6579219102859497, 캐모마일 티는 카페인이 없는 허브차로, 달지 않고 편안한 향을 즐기고 싶을 때 따뜻하게 마시기 좋습니다.
0.6229104995727539, 밀크티는 홍차와 우유를 섞은 부드러운 음료로, 달콤한 아이스 밀크티 또는 따뜻한 밀크티로 주문할 수 있습니다.
0.5817010402679443, 고구마 라테는 고구마의 달콤하고 고소한 맛을 살린 따뜻한 논커피 음료이며 카페인이 없습니다.


#### 다른 모델도 해보기 - jhgan/ko-sroberta-multitask

In [56]:
model2 = SentenceTransformer(
    "jhgan/ko-sroberta-multitask",
    device="cpu"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\study-with-ai-2\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--jhgan--ko-sroberta-multitask. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [60]:
cafe_vecs = model2.encode(
    sentences,
    normalize_embeddings=True
)
cafe_vecs.shape

(15, 768)

In [61]:
def search_v2(query, k=3):
    query_vec = model2.encode(
        query,
        normalize_embeddings=True
    )

    similarity = util.cos_sim(query_vec, cafe_vecs)[0]

    for i in similarity.argsort(descending=True)[:3]:
        i = int(i)
        print(f"{similarity[i]:.3f} | {sentences[i]}")

In [62]:
query1 = "카페인이 없고 따뜻한 음료 추천해 줘"
search_v2(query1)

0.759 | 초콜릿 라테는 진한 초콜릿과 우유로 만든 카페인 없는 달콤한 음료이며 따뜻한 겨울 음료로 좋습니다.
0.750 | 캐모마일 티는 카페인이 없는 허브차로, 달지 않고 편안한 향을 즐기고 싶을 때 따뜻하게 마시기 좋습니다.
0.712 | 고구마 라테는 고구마의 달콤하고 고소한 맛을 살린 따뜻한 논커피 음료이며 카페인이 없습니다.


In [63]:
query2 = "차갑고 달지 않은 커피가 뭐야?"
search_v2(query2)

0.748 | 디카페인 아메리카노는 카페인 부담을 줄인 커피로, 진하지만 달지 않은 맛을 원할 때 좋습니다.
0.743 | 콜드브루는 찬물로 오래 추출한 아이스 커피로, 산미가 적고 깔끔하며 시럽을 넣지 않아 달지 않습니다.
0.647 | 아메리카노는 에스프레소에 물을 더한 커피로, 뜨겁게 또는 차갑게 주문할 수 있으며 단맛이 거의 없습니다.


In [64]:
query3 = "달콤한 초콜릿 음료를 마시고 싶어"
search_v2(query3)

0.693 | 아이스 초코는 차가운 우유와 초콜릿 소스를 섞은 달콤한 논커피 음료로, 더운 날 시원하게 마시기 좋습니다.
0.655 | 초콜릿 라테는 진한 초콜릿과 우유로 만든 카페인 없는 달콤한 음료이며 따뜻한 겨울 음료로 좋습니다.
0.584 | 카라멜 마키아토는 우유와 바닐라 시럽 위에 에스프레소와 카라멜 소스를 올린 달콤한 따뜻한 커피입니다.


In [65]:
query4 = "따뜻한 티"
search_v2(query4)

0.577 | 자몽 허니 블랙티는 자몽과 꿀, 홍차를 넣은 달콤새콤한 논커피 음료로 아이스와 핫 모두 가능합니다.
0.561 | 캐모마일 티는 카페인이 없는 허브차로, 달지 않고 편안한 향을 즐기고 싶을 때 따뜻하게 마시기 좋습니다.
0.543 | 밀크티는 홍차와 우유를 섞은 부드러운 음료로, 달콤한 아이스 밀크티 또는 따뜻한 밀크티로 주문할 수 있습니다.


#### 또다른 모델로 해보기 - dragonkue/BGE-m3-ko

In [67]:
from sentence_transformers import SentenceTransformer

model3 = SentenceTransformer(
    "dragonkue/BGE-m3-ko",
    device="cpu"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [68]:
# batch_size=4 해야 한다

article_vecs = model3.encode(df['passage_add'].tolist()[:100], 
                             normalize_embeddings=True,
                             batch_size=4)


In [80]:
article_vecs.shape

(100, 1024)

In [69]:
def search_text3(query, k=3):    # 질문과 유사한 문서 최대 k 개 가져오기
    query_vec = model.encode("query: " + query, normalize_embeddings=True)
    similarity = util.cos_sim(query_vec, article_vecs)[0]
    for i in similarity.argsort(descending=True)[:k]:   #유사도 상위 k개만 뽑음
        i = int(i)
        print(f"{similarity[i]}, {df['정제본문'].iloc[i]}")



In [79]:
query = "러시아 관련해서 알려줘"
search_text3(query, 5)

0.3944745361804962, 이번 주 생글생글 761호 생글생글은 러시아 우크라이나 전쟁 이후 나타나고 있는 식량 대란 가능성을 분석했다 밀 수출 1 2위국의 전쟁이 세계 밀 공급을 크게 위축시키고 밀을 많이 생산하는 인도와 중국이 밀을 수출하지 않으려 해서 문제를 더 악화시키고 있는 점을 지적했다 맬서스 함정 수확체감의 법칙 애그플레이션이라는 용어도 만날 수 있다 유연근로가 필요한지 여부를 다룬 시사이슈 찬반토론도 유익하다
0.33799538016319275, 한은 우크라 사태 국내 물가 영향 원자재 해외 의존도 높은 산업 타격 러시아의 우크라이나 침공 중국 봉쇄 조치 등 최근 글로벌 공급망 차질이 기업의 비용 부담으로 이어져 이미 천정부지로 치솟는 물가 오름세가 더 커질 수 있다는 경고가 나왔다 공급망 차질이 지속되면 자동차와 2차전지 등 일부 업종은 생산에도 영향을 받을 것으로 보인다 한국은행은 4일 발표한 최근 글로벌 공급망 차질의 특징 및 국내 산업에 미치는 영향 보고서에서 우크라이나 사태 장기화 글로벌 식량 수급 불안 중국의 제로 코로나 정책 유지 등으로 향후 글로벌 공급망의 불확실성이 크다 며 이런 리스크가 현실화하면 대외 의존도가 높은 우리나라는 물가 오름세가 심화하고 생산에 대한 영향도 확대될 가능성이 있다 고 진단했다 보고서에 따르면 글로벌 공급망 차질로 인해 자동차 건설 기계장비 등 일부 산업은 부품 자재 수급 차질이 빚어져 생산이 일부 제약됐다 직접적인 생산 차질은 다른 국가들과 비교해 상대적으로 큰 편은 아니라는 게 한은의 설명이다 아울러 원자재 중간재 가격이 상승하면서 대부분 산업에서 비용 부담은 커졌다 실제로 생산단계별 물가를 보면 5월 기준으로 원재료는 1년 전보다 60 8 상승했고 중간재는 15 4 나 뛰었다 생산자물가 통계에서 공산품으로 분류된 품목 중 가격 상승률이 5 이상인 품목의 비중은 절반을 넘었고 가격 상승률이 10 이상인 품목은 약 40 에 달한다 생산자들이 제품을 만드는 데 드는 비용이 늘어난 만큼 앞으로 소